In [ ]:
# Import dependencies
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

In [ ]:
# Preview the case data
case_df = pd.read_csv("SCDB_2025_01_caseCentered_LegalProvision.csv")
case_df.head()

,caseId,docketId,caseIssuesId,voteId,dateDecision,decisionType,usCite,sctCite,ledCite,lexisCite,...,authorityDecision1,authorityDecision2,lawType,lawSupp,lawMinor,majOpinWriter,majOpinAssigner,splitVote,majVotes,minVotes
0,1946-001,1946-001-01,1946-001-01-01,1946-001-01-01-01,11/18/1946,1,329 U.S. 1,67 S. Ct. 6,91 L. Ed. 3,1946 U.S. LEXIS 1724,...,4.0,NaN,6.0,600.0,35 U.S.C. § 33,78.0,78.0,1,8,1
1,1946-002,1946-002-01,1946-002-01-01,1946-002-01-01-01,11/18/1946,1,329 U.S. 14,67 S. Ct. 13,91 L. Ed. 12,1946 U.S. LEXIS 1725,...,4.0,NaN,6.0,600.0,18 U.S.C. § 398,81.0,87.0,1,6,3
2,1946-002,1946-002-02,1946-002-02-01,1946-002-02-01-01,11/18/1946,1,329 U.S. 14,67 S. Ct. 13,91 L. Ed. 12,1946 U.S. LEXIS 1725,...,4.0,NaN,6.0,600.0,18 U.S.C. § 398,81.0,87.0,1,6,3
3,1946-002,1946-002-03,1946-002-03-01,1946-002-03-01-01,11/18/1946,1,329 U.S. 14,67 S. Ct. 13,91 L. Ed. 12,1946 U.S. LEXIS 1725,...,4.0,NaN,6.0,600.0,18 U.S.C. § 398,81.0,87.0,1,6,3
4,1946-002,1946-002-04,1946-002-04-01,1946-002-04-01-01,11/18/1946,1,329 U.S. 14,67 S. Ct. 13,91 L. Ed. 12,1946 U.S. LEXIS 1725,...,4.0,NaN,6.0,600.0,18 U.S.C. § 398,81.0,87.0,1,6,3


In [ ]:
# Store the dimensions
dimensions = case_df.shape
num_rows = dimensions[0]
num_cols = dimensions[1]

# Report counts of missing values per column.
print("")
print("Number of Missing Values per Column:")
for col in case_df.columns:
    print(col + ": " + str(case_df[col].isna().sum()))
print("")

# Create a mapping of the columns to the corresponding percentages of values missing.
cols_to_percentage_missing = {}

# Print the percentage of values missing per column.
print("Percentage of Values Missing per Column:")
for col in case_df.columns:
    print(col + ": " + str(100*case_df[col].isna().sum()/num_rows) + "%")
    cols_to_percentage_missing[col] = 100*case_df[col].isna().sum()/num_rows
print("")

# Remove the columns whose percentages of values missing exceed 20%
for col, percentage in cols_to_percentage_missing.items():
    if percentage > 20:
        case_df.drop(columns=[col], inplace=True)

# Preview the new DataFrame
print("Preview of DataFrame with Columns Removed:")
print(case_df.head())
print("")

# Report the data types of the variables in the new DataFrame.
print("Data Types in the new DataFrame:")
print(case_df.dtypes)


Number of Missing Values per Column:
caseId: 0
docketId: 0
caseIssuesId: 0
voteId: 0
dateDecision: 0
decisionType: 0
usCite: 604
sctCite: 7
ledCite: 6
lexisCite: 0
term: 0
naturalCourt: 0
chief: 0
docket: 31
caseName: 0
dateArgument: 1326
dateRearg: 13558
petitioner: 7
petitionerState: 11198
respondent: 14
respondentState: 10145
jurisdiction: 5
adminAction: 9733
adminActionState: 12855
threeJudgeFdc: 32
caseOrigin: 480
caseOriginState: 10239
caseSource: 291
caseSourceState: 10844
lcDisagreement: 32
certReason: 132
lcDisposition: 2272
lcDispositionDirection: 262
declarationUncon: 1
caseDisposition: 164
caseDispositionUnusual: 2
partyWinning: 17
precedentAlteration: 1
voteUnclear: 3
issue: 72
issueArea: 72
decisionDirection: 47
decisionDirectionDissent: 364
authorityDecision1: 83
authorityDecision2: 11666
lawType: 1636
lawSupp: 1636
lawMinor: 11388
majOpinWriter: 2249
majOpinAssigner: 343
splitVote: 0
majVotes: 0
minVotes: 0

Percentage of Values Missing per Column:
caseId: 0.0%
docketI

In [ ]:
"""
The goal of this project is for users (likely the lawyers of the losing party
from a case at a Court of Appeals) to input information regarding their case and
to output a prediction on whether, even if the Supreme Court agrees to the review
their case, they would prevail or not. Therefore, we need to make sure that
the models are only trained on information that the users would realistically know
at the time of using this application. That being said, we need to remove any post-case
details from the collection of features to train on.
"""
after_case_details = ["decisionType", "dateDecision", "declarationUncon", "caseDisposition",
                      "caseDispositionUnusual", "precedentAlteration", "voteUnclear",
                      "decisionDirection", "decisionDirectionDissent", "authorityDecision1", "majOpinWriter",
                      "majOpinAssigner", "splitVote", "majVotes", "minVotes", "certReason", "dateArgument",
                      "usCite", "sctCite", "ledCite", "lexisCite", "docket", "caseName"]
case_df.drop(columns=after_case_details, inplace=True)

# Preview the new DataFrame
print("Preview of DataFrame with Columns Removed:")
print(case_df.head())
print("")

# Report the data types of the variables in the new DataFrame.
print("Data Types in the new DataFrame:")
print(case_df.dtypes)

Preview of DataFrame with Columns Removed:
     caseId     docketId    caseIssuesId             voteId  term  \
0  1946-001  1946-001-01  1946-001-01-01  1946-001-01-01-01  1946   
1  1946-002  1946-002-01  1946-002-01-01  1946-002-01-01-01  1946   
2  1946-002  1946-002-02  1946-002-02-01  1946-002-02-01-01  1946   
3  1946-002  1946-002-03  1946-002-03-01  1946-002-03-01-01  1946   
4  1946-002  1946-002-04  1946-002-04-01  1946-002-04-01-01  1946   

   naturalCourt   chief  petitioner  respondent  jurisdiction  ...  \
0          1301  Vinson       198.0       172.0           6.0  ...   
1          1301  Vinson       100.0        27.0           1.0  ...   
2          1301  Vinson       100.0        27.0           1.0  ...   
3          1301  Vinson       100.0        27.0           1.0  ...   
4          1301  Vinson       100.0        27.0           1.0  ...   

   caseOrigin  caseSource  lcDisagreement  lcDisposition  \
0        51.0        29.0             0.0            2.0   
1

In [ ]:
"""
Remove identifiers from the data (except for caseId, which will be important for
splitting into train and test data as we will later see).
"""
identifiers = ["docketId", "caseIssuesId", "voteId"]
case_df.drop(columns=identifiers, inplace=True)

# Preview the new DataFrame
print("Preview of DataFrame with Columns Removed:")
print(case_df.head())
print("")

# Report the data types of the variables in the new DataFrame.
print("Data Types in the new DataFrame:")
print(case_df.dtypes)
print("")

Preview of DataFrame with Columns Removed:
     caseId  term  naturalCourt   chief  petitioner  respondent  jurisdiction  \
0  1946-001  1946          1301  Vinson       198.0       172.0           6.0   
1  1946-002  1946          1301  Vinson       100.0        27.0           1.0   
2  1946-002  1946          1301  Vinson       100.0        27.0           1.0   
3  1946-002  1946          1301  Vinson       100.0        27.0           1.0   
4  1946-002  1946          1301  Vinson       100.0        27.0           1.0   

   threeJudgeFdc  caseOrigin  caseSource  lcDisagreement  lcDisposition  \
0            0.0        51.0        29.0             0.0            2.0   
1            0.0       123.0        30.0             0.0            2.0   
2            0.0       123.0        30.0             0.0            2.0   
3            0.0       123.0        30.0             0.0            2.0   
4            0.0       123.0        30.0             0.0            2.0   

   lcDispositionDir

In [ ]:
"""
Remove the chief justices column since the only value provided by this feature in
the context of this dataset is when the case happened (already captured by the
term feature) since there is no further information regarding their political affiliation.
"""
case_df.drop(columns=["chief"], inplace=True)

# Preview the new DataFrame
print("Preview of DataFrame with Columns Removed:")
print(case_df.head())
print("")

# Report the data types of the variables in the new DataFrame.
print("Data Types in the new DataFrame:")
print(case_df.dtypes)

# Report counts of missing values per column.
print("")
print("Number of Missing Values per Column:")
for col in case_df.columns:
    print(col + ": " + str(case_df[col].isna().sum()))
print("")

# Create a mapping of the columns to the corresponding percentages of values missing.
cols_to_percentage_missing = {}

# Print the percentage of values missing per column.
print("Percentage of Values Missing per Column:")
for col in case_df.columns:
    print(col + ": " + str(100*case_df[col].isna().sum()/num_rows) + "%")
    cols_to_percentage_missing[col] = 100*case_df[col].isna().sum()/num_rows
print("")

Preview of DataFrame with Columns Removed:
     caseId  term  naturalCourt  petitioner  respondent  jurisdiction  \
0  1946-001  1946          1301       198.0       172.0           6.0   
1  1946-002  1946          1301       100.0        27.0           1.0   
2  1946-002  1946          1301       100.0        27.0           1.0   
3  1946-002  1946          1301       100.0        27.0           1.0   
4  1946-002  1946          1301       100.0        27.0           1.0   

   threeJudgeFdc  caseOrigin  caseSource  lcDisagreement  lcDisposition  \
0            0.0        51.0        29.0             0.0            2.0   
1            0.0       123.0        30.0             0.0            2.0   
2            0.0       123.0        30.0             0.0            2.0   
3            0.0       123.0        30.0             0.0            2.0   
4            0.0       123.0        30.0             0.0            2.0   

   lcDispositionDirection  partyWinning    issue  issueArea  lawTyp

In [ ]:
"""
Fill lcDisposition, lawType, and lawSupp columns with constants denoting missing values
since they have missing percentages of 11%-17%
"""
case_df["lcDisposition"].fillna(-10, inplace=True)
case_df["lawType"].fillna(-10, inplace=True)
case_df["lawSupp"].fillna(-1000, inplace=True)

/tmp/ipykernel_6903/1844642597.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  case_df["lcDisposition"].fillna(-10, inplace=True)
/tmp/ipykernel_6903/1844642597.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

In [ ]:
# Remove any observations where there any of the feature values are missing.
for col in case_df.columns:
    case_df = case_df[case_df[col].notna()]

# Preview the new DataFrame
print("Preview of DataFrame with Columns Removed:")
print(case_df.head())
print("")

# Report the data types of the variables in the new DataFrame.
print("Data Types in the new DataFrame:")
print(case_df.dtypes)

# Report counts of missing values per column.
print("")
print("Number of Missing Values per Column:")
for col in case_df.columns:
    print(col + ": " + str(case_df[col].isna().sum()))
print("")

# Create a mapping of the columns to the corresponding percentages of values missing.
cols_to_percentage_missing = {}

# Print the percentage of values missing per column.
print("Percentage of Values Missing per Column:")
for col in case_df.columns:
    print(col + ": " + str(100*case_df[col].isna().sum()/num_rows) + "%")
    cols_to_percentage_missing[col] = 100*case_df[col].isna().sum()/num_rows
print("")

Preview of DataFrame with Columns Removed:
     caseId  term  naturalCourt  petitioner  respondent  jurisdiction  \
0  1946-001  1946          1301       198.0       172.0           6.0   
1  1946-002  1946          1301       100.0        27.0           1.0   
2  1946-002  1946          1301       100.0        27.0           1.0   
3  1946-002  1946          1301       100.0        27.0           1.0   
4  1946-002  1946          1301       100.0        27.0           1.0   

   threeJudgeFdc  caseOrigin  caseSource  lcDisagreement  lcDisposition  \
0            0.0        51.0        29.0             0.0            2.0   
1            0.0       123.0        30.0             0.0            2.0   
2            0.0       123.0        30.0             0.0            2.0   
3            0.0       123.0        30.0             0.0            2.0   
4            0.0       123.0        30.0             0.0            2.0   

   lcDispositionDirection  partyWinning    issue  issueArea  lawTyp

In [ ]:
# Determine the number of outcomes for partyWinning.
partyWinning_outcomes = np.unique(case_df["partyWinning"])
print("partyWinning outcomes:", partyWinning_outcomes)

partyWinning outcomes: [0. 1. 2.]


In [ ]:
# Print the frequencies of each outcome for partyWinning.
for outcome in partyWinning_outcomes:
    partyWinningCol = case_df["partyWinning"]
    print(f"Frequency of {outcome}: {partyWinningCol[partyWinningCol == outcome].size}")

Frequency of 0.0: 4939
Frequency of 1.0: 8421
Frequency of 2.0: 6


In [ ]:
"""
Since the frequency of outcome 2 (unclear who the favorable party is) is very small,
remove all observations corresponding to this outcome.
"""
case_df = case_df[(case_df["partyWinning"] == 0.0) | (case_df["partyWinning"] == 1.0)]
partyWinning_outcomes = np.unique(case_df["partyWinning"])

# Reprint the outcomes for partyWinning to guarantee the observations were removed.
print("partyWinning outcomes:", partyWinning_outcomes)

partyWinning outcomes: [0. 1.]


In [ ]:
# Remove any observations where there any of the feature values are missing.
for col in case_df.columns:
    case_df = case_df[case_df[col].notna()]

# Preview the new DataFrame
print("Preview of DataFrame with Columns Removed:")
print(case_df.head())
print("")

# Report the data types of the variables in the new DataFrame.
print("Data Types in the new DataFrame:")
print(case_df.dtypes)

# Report counts of missing values per column.
print("")
print("Number of Missing Values per Column:")
for col in case_df.columns:
    print(col + ": " + str(case_df[col].isna().sum()))
print("")

# Create a mapping of the columns to the corresponding percentages of values missing.
cols_to_percentage_missing = {}

# Print the percentage of values missing per column.
print("Percentage of Values Missing per Column:")
for col in case_df.columns:
    print(col + ": " + str(100*case_df[col].isna().sum()/num_rows) + "%")
    cols_to_percentage_missing[col] = 100*case_df[col].isna().sum()/num_rows
print("")

Preview of DataFrame with Columns Removed:
     caseId  term  naturalCourt  petitioner  respondent  jurisdiction  \
0  1946-001  1946          1301       198.0       172.0           6.0   
1  1946-002  1946          1301       100.0        27.0           1.0   
2  1946-002  1946          1301       100.0        27.0           1.0   
3  1946-002  1946          1301       100.0        27.0           1.0   
4  1946-002  1946          1301       100.0        27.0           1.0   

   threeJudgeFdc  caseOrigin  caseSource  lcDisagreement  lcDisposition  \
0            0.0        51.0        29.0             0.0            2.0   
1            0.0       123.0        30.0             0.0            2.0   
2            0.0       123.0        30.0             0.0            2.0   
3            0.0       123.0        30.0             0.0            2.0   
4            0.0       123.0        30.0             0.0            2.0   

   lcDispositionDirection  partyWinning    issue  issueArea  lawTyp

In [ ]:
"""
# Display the correlation matrix of the current dataset to spot possibly redundant features.
correlation_matrix = case_df.drop(columns=["caseId"]).corr()
print(correlation_matrix)
sns.heatmap(correlation_matrix)
"""

'\n# Display the correlation matrix of the current dataset to spot possibly redundant features.\ncorrelation_matrix = case_df.drop(columns=["caseId"]).corr()\nprint(correlation_matrix)\nsns.heatmap(correlation_matrix)\n'

In [ ]:
"""
# Print a mutual information matrix for the features in relation to the target variable (partyWinning)
mutual_info_matrix = mutual_info_classif(case_df.drop(columns=["caseId", "partyWinning"]), case_df["partyWinning"], random_state=0)

# Print the mutual information matrix (found from research that it assesses any kind of relationship - linear or nonlinear).
print("Mutual Information Matrix")
print(mutual_info_matrix)
"""

'\n# Print a mutual information matrix for the features in relation to the target variable (partyWinning)\nmutual_info_matrix = mutual_info_classif(case_df.drop(columns=["caseId", "partyWinning"]), case_df["partyWinning"], random_state=0)\n\n# Print the mutual information matrix (found from research that it assesses any kind of relationship - linear or nonlinear).\nprint("Mutual Information Matrix")\nprint(mutual_info_matrix)\n'

In [ ]:
"""
The following pairs of features were found to be highly correlated with each other:
1) Term and Natural Court
2) Case Origin and Case Source
3) Issue and Issue Area
4) Law Type and Law Supp
"""

"""
Select the features from each pair to remove based on the mutual information matrix
values (if a feature has a higher value in the mutual information matrix than another,
discard the feature with the lower value).
"""

"""
highly_correlated_features_to_remove = ["term", "caseSource", "issue", "lawType"]

# Remove these features
case_df.drop(columns=highly_correlated_features_to_remove, inplace=True)

# Preview the new DataFrame
print("Preview of DataFrame with Columns Removed:")
print(case_df.head())
print("")

# Report the data types of the variables in the new DataFrame.
print("Data Types in the new DataFrame:")
print(case_df.dtypes)
print("")
"""

'\nhighly_correlated_features_to_remove = ["term", "caseSource", "issue", "lawType"]\n\n# Remove these features\ncase_df.drop(columns=highly_correlated_features_to_remove, inplace=True)\n\n# Preview the new DataFrame\nprint("Preview of DataFrame with Columns Removed:")\nprint(case_df.head())\nprint("")\n\n# Report the data types of the variables in the new DataFrame.\nprint("Data Types in the new DataFrame:")\nprint(case_df.dtypes)\nprint("")\n'

In [ ]:
# Save the current DataFrame as a csv file to use for the model training pipeline.
case_df.to_csv("preprocessed_case_data.csv", index=False)